
# Introduction to Basic Quantum Gate Operations with Qiskit

This notebook introduces the most common quantum gates used in quantum computing.  
It is designed for beginners and senior high school students who are learning the connection between **quantum states**, **quantum gates**, and **measurement outcomes**.

Jerry Chen, NTU-IBM Quantum Hub.

We will cover:

1. Single-qubit states: $|0\rangle, |1\rangle$, and superposition  
2. Basic single-qubit gates: X, H, Z, S, T, Rx, Ry, Rz  
3. Measurement and probability  
4. Two-qubit gates: CNOT, CZ, SWAP  
5. Creating entanglement with H + CNOT  
6. Comparing statevector simulation and shot-based measurement



## 0. Install and Import Qiskit

If Qiskit is not installed, run:

```bash
pip install qiskit qiskit-aer matplotlib pylatexenc
```

`pylatexenc` is optional but useful for drawing circuits nicely.


In [ ]:
# Basic imports
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram, plot_bloch_multivector

print('Qiskit imports loaded successfully.')


## 1. Quantum State Basics

A classical bit can only be 0 or 1.

A single qubit can be written as:


$|\psi\rangle = \alpha |0\rangle + \beta |1\rangle$

where $\alpha$ and $\beta$ are complex amplitudes, and:

$|\alpha|^2$ + $|\beta|^2 = 1$

After measurement:

- Probability of measuring 0 is $|\alpha|^2$
- Probability of measuring 1 is $|\beta|^2$


In [ ]:
# Start with the |0> state
qc = QuantumCircuit(1)
state = Statevector.from_instruction(qc)

print('Statevector for |0>:')
print(state)

qc.draw('mpl')

In [ ]:
plot_bloch_multivector(state)


## 2. X Gate: Quantum NOT Gate

The X gate flips $|0\rangle$ to $|1\rangle$, and $|1\rangle$ to $|0\rangle$.

Matrix form:

$$ X = \begin{bmatrix}0 & 1 \\ 1 & 0\end{bmatrix}$$

Action:

$$X|0\rangle = |1\rangle, \quad X|1\rangle = |0\rangle$$


In [ ]:
qc = QuantumCircuit(1)
qc.___(0)  # TODO: Apply an X gate

state = Statevector.from_instruction(qc)
print('Statevector after X gate:')
print(state)

qc.draw('mpl')

In [ ]:
plot_bloch_multivector(state)


## 3. H Gate: Creating Superposition

The Hadamard gate maps $|0\rangle$ into an equal superposition:

$$H|0\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}}$$

This means measurement gives 0 or 1 with approximately 50% probability each.

Matrix form:

$$H = \frac{1}{\sqrt{2}}\begin{bmatrix}1 & 1 \\ 1 & -1\end{bmatrix}$$


In [ ]:
qc = QuantumCircuit(1)
qc.___(0)  # TODO: Apply a Hadamard gate

state = Statevector.from_instruction(qc)
print('Statevector after H gate:')
print(state)

qc.draw('mpl')

In [ ]:
plot_bloch_multivector(state)


### Measuring the H Gate Result

Statevector tells us the exact quantum state.  
However, real quantum computers only give measurement outcomes.

Here we simulate repeated measurements using 1024 shots.


In [ ]:
qc = QuantumCircuit(1, 1)
qc.___(0)  # TODO: Apply a Hadamard gate
qc.measure(0, 0)

simulator = AerSimulator()
compiled = transpile(qc, simulator)
result = simulator.run(compiled, shots=1024).result()
counts = result.get_counts()

print(counts)
plot_histogram(counts)


## 4. Z Gate: Phase Flip

The Z gate does not change $|0\rangle$, but it changes the sign of $|1\rangle$:

$$Z|0\rangle = |0\rangle, \quad Z|1\rangle = -|1\rangle$$

Matrix form:

$$Z = \begin{bmatrix}1 & 0 \\ 0 & -1\end{bmatrix}$$

Important idea:  
If the qubit is only $|0\rangle$ or only $|1\rangle$, a global sign may not change measurement probabilities.  
But when the state is a superposition, phase differences can affect later interference.


In [ ]:
# Compare H only vs H-Z-H
qc1 = QuantumCircuit(1)
qc1.h(0)
state1 = Statevector.from_instruction(qc1)

qc2 = QuantumCircuit(1)
qc2.h(0)
qc2.z(0)
qc2.h(0)
state2 = Statevector.from_instruction(qc2)

print('State after H:')
print(state1)
print('State after H-Z-H:')
print(state2)

qc2.draw('mpl')


The circuit H-Z-H produces the same effect as an X gate.  
This shows that a change in quantum phase can become observable through interference.


In [ ]:
plot_bloch_multivector(state1)

In [ ]:
plot_bloch_multivector(state2)


## 5. S and T Gates: Phase Rotation Gates

The S and T gates rotate the phase of the $|1\rangle$ component.

$$S = \begin{bmatrix}1 & 0 \\ 0 & i\end{bmatrix}$$

$$T = \begin{bmatrix}1 & 0 \\ 0 & e^{i\pi/4}\end{bmatrix}$$

They are important because quantum algorithms often use phase differences to create interference.

| Gate | Bloch-sphere rotation | Effect on relative phase |
|:----:|:----------------------|:-------------------------|
| $S$ | $Z$-axis by $90^\circ$ | $|1\rangle \rightarrow i|1\rangle$ |
| $T$ | $Z$-axis by $45^\circ$ | $|1\rangle \rightarrow e^{i\pi/4}|1\rangle$ |

In [ ]:
for gate_name in ['s', 't']:
    qc = QuantumCircuit(1)
    qc.___(0)  # TODO: Apply a Hadamard gate
    getattr(qc, gate_name)(0) # equivalent to qc.s(0) and then qc.t(0) in each loop
    state = Statevector.from_instruction(qc)
    print(f'State after H + {gate_name.upper()}:')
    print(state)
    display(qc.draw('mpl'))
    display(plot_bloch_multivector(state))


## 6. Rotation Gates: Rx, Ry, and Rz

Rotation gates rotate the qubit state around the Bloch sphere axes.

- $R_x(\theta)$: rotation around the x-axis
- $R_y(\theta)$: rotation around the y-axis
- $R_z(\theta)$: rotation around the z-axis

For example, $R_y(\pi/2)|0\rangle$ creates a 50-50 superposition similar to the H gate, but with a different geometric interpretation.


In [ ]:
theta = np.pi / 2

qc = QuantumCircuit(1)
qc.ry(theta, 0)
state = Statevector.from_instruction(qc)

print('State after Ry(pi/2):')
print(state)

qc.draw('mpl')

In [ ]:
plot_bloch_multivector(state)


## 7. CNOT Gate: Controlled NOT

CNOT is a two-qubit gate.

- The first qubit is the control qubit.
- The second qubit is the target qubit.
- If the control qubit is $|1\rangle$, the target qubit flips.
- If the control qubit is $|0\rangle$, the target qubit stays the same.

Truth table:

| Input | Output |
|---|---|
| 00 | 00 |
| 01 | 11 |
| 10 | 10 |
| 11 | 01 |

Note:
In Qiskit, qubits are ordered using little-endian notation, with the least significant qubits having smaller indices.


In [ ]:
# Example: input |01>, then CNOT should produce |11>
qc = QuantumCircuit(2)
qc.___(0)  # TODO: Apply an X gate       # Prepare q0 = 1
qc.cx(0, 1)   # CNOT: q0 controls q1

state = Statevector.from_instruction(qc)
print(state)

qc.draw('mpl')

In [ ]:
qc = QuantumCircuit(2)
qc.___(0)  # TODO: Apply an X gate       # Prepare q0 = 1
qc.cx(0, 1)   # CNOT: q0 controls q1
qc.measure_all()

simulator = AerSimulator()
compiled = transpile(qc, simulator)
result = simulator.run(compiled, shots=1024).result()
counts = result.get_counts()

print(counts)
plot_histogram(counts)


## 8. Creating Entanglement with H + CNOT

A very important two-qubit circuit is:

1. Apply H to the first qubit.
2. Apply CNOT from the first qubit to the second qubit.

This creates the Bell state:

$$\frac{|00\rangle + |11\rangle}{\sqrt{2}}$$

This is an entangled state.  
The two qubits are not independent anymore: if we measure one as 0, the other is also 0; if we measure one as 1, the other is also 1.


In [ ]:
qc = QuantumCircuit(2)
qc.___(0)  # TODO: Apply a Hadamard gate
qc.cx(0, 1)

state = Statevector.from_instruction(qc)
print('Bell state:')
print(state)

qc.draw('mpl')

In [ ]:
plot_bloch_multivector(state)

In [ ]:
qc = QuantumCircuit(2, 2)
qc.___(0)  # TODO: Apply a Hadamard gate
qc.cx(0, 1)
qc.measure([0, 1], [0, 1])

simulator = AerSimulator()
compiled = transpile(qc, simulator)
result = simulator.run(compiled, shots=1024).result()
counts = result.get_counts()

print(counts)
plot_histogram(counts)


## 9. CZ Gate: Controlled Phase Flip

CZ is similar to CNOT, but instead of flipping the target qubit, it flips the phase of the $|11\rangle$ state.

$CZ|11\rangle = -|11\rangle$

All other computational basis states stay the same.

CZ is very important in quantum algorithms because many algorithms encode information into phase.


In [ ]:
qc = QuantumCircuit(2)
qc.___(0)  # TODO: Apply a Hadamard gate
qc.h(1)
qc.cz(0, 1)

state = Statevector.from_instruction(qc)
print('State after H on both qubits and CZ:')
print(state)

qc.draw('mpl')

In [ ]:
plot_bloch_multivector(state)

In [ ]:
qc = QuantumCircuit(2, 2)
qc.___(0)  # TODO: Apply a Hadamard gate
qc.h(1)
qc.cz(0, 1)
qc.measure([0, 1], [0, 1]) # Measure qubits 0 and 1, storing their results in classical bits 0 and 1, respectively.

simulator = AerSimulator()
compiled = transpile(qc, simulator)
result = simulator.run(compiled, shots=1024).result()
counts = result.get_counts()

print(counts)
plot_histogram(counts)


## 10. SWAP Gate

The SWAP gate exchanges the states of two qubits.

$$SWAP|a,b\rangle = |b,a\rangle$$

It is often used when a real quantum computer does not allow a direct two-qubit gate between two distant physical qubits.


In [ ]:
# Prepare |01> then swap it to |10>
qc = QuantumCircuit(2)
qc.x(1)
qc.___(0, 1)  # TODO: Apply a SWAP gate

state = Statevector.from_instruction(qc)
print(state)

qc.draw('mpl')

In [ ]:
plot_bloch_multivector(state)

In [ ]:
qc = QuantumCircuit(2, 2)
qc.x(1)
qc.___(0, 1)  # TODO: Apply a SWAP gate
qc.measure([0, 1], [0, 1]) # Measure qubits 0 and 1, storing their results in classical bits 0 and 1, respectively.

simulator = AerSimulator()
compiled = transpile(qc, simulator)
result = simulator.run(compiled, shots=1024).result()
counts = result.get_counts()

print(counts)
plot_histogram(counts)


## 11. Mini Practice: Predict Before Running

Try to predict the measurement result before running the code.

Circuit:

1. Start from $|0\rangle$
2. Apply H
3. Apply Z
4. Apply H
5. Measure

Question: will the output be mostly 0, mostly 1, or 50-50?


In [ ]:
qc = QuantumCircuit(1, 1)
qc.___(0)  # TODO: Apply a Hadamard gate
qc.z(0)
qc.h(0)
qc.measure(0, 0)

simulator = AerSimulator()
result = simulator.run(transpile(qc, simulator), shots=1024).result()
counts = result.get_counts()

print(counts)
plot_histogram(counts)


## 12. Summary

In this notebook, we introduced the basic quantum gates:

| Gate | Meaning |
|---|---|
| X | Bit flip, like classical NOT |
| H | Creates superposition |
| Z | Phase flip |
| S, T | Phase rotations |
| Rx, Ry, Rz | Bloch sphere rotations |
| CNOT | Controlled bit flip |
| CZ | Controlled phase flip |
| SWAP | Exchange two qubit states |

Key ideas:

- Quantum gates change amplitudes and phases.
- Measurement turns a quantum state into classical outcomes.
- Superposition gives probabilistic outcomes.
- Phase may be invisible immediately, but it affects interference.
- Entanglement creates strong correlations between qubits.
